
# 🚀 Advanced AML Name Screening & Analyst Workload Optimization Platform

This notebook implements a COMPLETE end-to-end AML prototype including:

1. Synthetic Data Generation  
2. Feature Engineering  
3. ML Model Training (Logistic Regression)  
4. Model Evaluation  
5. Risk Scoring Engine  
6. Analyst Workload Prioritization  
7. Schema-Adaptive Dataset Detection  
8. Unified Dataset Risk Engine  
9. Explainability (Feature Importance)  

This is a full working AML project inside a single notebook.


In [ ]:
!pip install rapidfuzz scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:

import pandas as pd
import numpy as np
from rapidfuzz import fuzz
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib


## 🔹 Feature Engineering Functions

In [ ]:

def normalize(text):
    return text.lower().strip()

def name_similarity(a, b):
    return fuzz.token_sort_ratio(a, b) / 100.0

def dob_similarity(d1, d2):
    return 1.0 if d1 == d2 else 0.0

def country_match(c1, c2):
    return 1.0 if c1 == c2 else 0.0

def gender_match(g1, g2):
    return 1.0 if g1 == g2 else 0.0

def generate_features(customer, watchlist):
    return [
        name_similarity(normalize(customer["name"]), normalize(watchlist["name"])),
        dob_similarity(customer["dob"], watchlist["dob"]),
        country_match(customer["country"], watchlist["country"]),
        gender_match(customer["gender"], watchlist["gender"]),
        watchlist["severity"]
    ]


## 🔹 Synthetic Training Data Generation

In [ ]:

np.random.seed(42)

def random_similarity():
    return np.random.uniform(0.4, 1.0)

data = []
labels = []

for _ in range(500):
    name_sim = random_similarity()
    dob = np.random.choice([0,1], p=[0.7,0.3])
    country = np.random.choice([0,1], p=[0.6,0.4])
    gender = np.random.choice([0,1], p=[0.5,0.5])
    severity = np.random.uniform(0.3,1.0)
    
    # True match rule simulation
    label = 1 if (name_sim > 0.8 and dob==1 and country==1) else 0
    
    data.append([name_sim,dob,country,gender,severity])
    labels.append(label)

X = np.array(data)
y = np.array(labels)

print("Dataset shape:", X.shape)


## 🔹 Train ML Model

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

model = LogisticRegression()
model.fit(X_train,y_train)

preds = model.predict(X_test)

print(classification_report(y_test,preds))


## 🔹 Confusion Matrix

In [ ]:

cm = confusion_matrix(y_test,preds)
sns.heatmap(cm,annot=True,fmt="d")
plt.title("Confusion Matrix")
plt.show()


In [ ]:

joblib.dump(model,"aml_risk_model.pkl")
print("Model saved.")


## 🔹 Risk Prediction Engine

In [ ]:

def predict_risk(features):
    return model.predict_proba([features])[0][1]

def decision_logic(prob):
    if prob > 0.8:
        return "HIGH PRIORITY REVIEW"
    elif prob > 0.4:
        return "MEDIUM PRIORITY REVIEW"
    else:
        return "LOW PRIORITY / AUTO CLOSE"


## 🔹 Test Real Case

In [ ]:

customer = {
    "name":"Mohammad Ali",
    "dob":"1995-04-10",
    "country":"India",
    "gender":"Male"
}

watchlist = {
    "name":"Muhammad Ali",
    "dob":"1960-01-01",
    "country":"Syria",
    "gender":"Male",
    "severity":0.9
}

features = generate_features(customer, watchlist)
prob = predict_risk(features)
decision = decision_logic(prob)

print("Features:",features)
print("True Match Probability:",prob)
print("Decision:",decision)


## 🔹 Schema-Adaptive Column Detection

In [ ]:

ONTOLOGY = {
    "amount": ["amount","txn_amt","value"],
    "country": ["country","ctry"],
    "name": ["name","customer_name"]
}

def detect_columns(df):
    mapping = {}
    for col in df.columns:
        for key, candidates in ONTOLOGY.items():
            for candidate in candidates:
                if fuzz.partial_ratio(col.lower(), candidate.lower()) > 80:
                    mapping[key] = col
    return mapping

sample_data = pd.DataFrame({
    "cust_name":["Ali","Rahul"],
    "txn_amt":[120000,50000],
    "country":["Iran","India"]
})

mapping = detect_columns(sample_data)
print("Detected Mapping:",mapping)


## 🔹 Unified Dataset Risk Engine

In [ ]:

def unified_risk(df, mapping):
    scores=[]
    if "amount" in mapping:
        avg=df[mapping["amount"]].mean()
        scores.append(0.9 if avg>100000 else 0.3)
    if "country" in mapping:
        high=["Iran","Syria","North Korea"]
        scores.append(0.8 if df[mapping["country"]].isin(high).any() else 0.2)
    return np.mean(scores)

print("Unified Dataset Risk:",unified_risk(sample_data,mapping))


## 🔹 Feature Importance (Explainability)

In [ ]:

importance = model.coef_[0]
feature_names = ["NameSim","DOB","Country","Gender","Severity"]

plt.barh(feature_names,importance)
plt.title("Feature Importance")
plt.show()
